In [6]:
import pandas as pd
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction import DictVectorizer
import numpy as np
import time

In [7]:
# Read January 2023 data
df_jan = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')

# Read Feb 2023 data
df_feb = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

print(f"Number of columns: {len(df_jan.columns)}")
print(df_jan.columns.tolist())

Number of columns: 19
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [8]:
# Compute duration in minutes
df_jan['duration'] = (df_jan['tpep_dropoff_datetime'] - df_jan['tpep_pickup_datetime']).dt.total_seconds() / 60

# Calculate standard deviation
std_duration = df_jan['duration'].std()
print(f"standard deviation of duration: {std_duration:.2f} minutes")

standard deviation of duration: 42.59 minutes


In [9]:
# Count original records
original_count = len(df_jan)

# Filter: keep only durations between 1 and 60 minutes (inclusive)
df_jan_filtered = df_jan[(df_jan['duration'] >= 1) & (df_jan['duration'] <= 60)].copy()

# Count filtered records
filtered_count = len(df_jan_filtered)

# Calculate fraction
fraction_remaining = filtered_count / original_count
print(f"Original records: {original_count}")
print(f"Records after filtering: {filtered_count}")
print(f"Fraction remaining: {fraction_remaining:.2f}")

Original records: 3066766
Records after filtering: 3009173
Fraction remaining: 0.98


In [10]:
# Select only pickup and dropoff location IDs, cast to strings
df_jan_filtered.loc[:,'PULocationID'] = df_jan_filtered['PULocationID'].astype(str)
df_jan_filtered.loc[:,'DOLocationID'] = df_jan_filtered['DOLocationID'].astype(str)

# Turn dataframe into list of dictionaries
dicts = df_jan_filtered[['PULocationID', 'DOLocationID']].to_dict(orient='records')

# Fit a DictVectorizer
dv = DictVectorizer(sparse=True)
X = dv.fit_transform(dicts)

# Check dimensionality
print(f"Shape of feature matrix: {X.shape}")
print(f"Number of columns (features): {X.shape[1]}")

Shape of feature matrix: (3009173, 515)
Number of columns (features): 515


In [11]:
# Prepare target variable
y_train = df_jan_filtered['duration'].values

# Train linear regression model
lr = LinearRegression()
lr.fit(X, y_train)

# Predict on training data
y_pred = lr.predict(X)

# Calculate RMSE
rmse = mean_squared_error(y_train, y_pred, squared=False)
#rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"RMSE on training dara: {rmse:.2f}")

RMSE on training dara: 7.65


In [8]:
# Compute duration in minutes
df_feb['duration'] = (df_feb['tpep_dropoff_datetime'] - df_feb['tpep_pickup_datetime']).dt.total_seconds() / 60

# Filter outliers
df_feb_filtered = df_feb[(df_feb['duration'] >= 1) & (df_feb['duration'] <= 60)]
print(f"Februrary records after filtering: {len(df_feb_filtered)}")

#Prepare features (convert to string for one-hot encoding)
dicts_val = df_feb_filtered[['PULocationID', 'DOLocationID']].astype(str).to_dict(orient='records')

#Transform using the SAME DictVectorizer fitted on training data (don't fit again!)
print("Transforming features..")
X_val = dv.transform(dicts_val)
print(f"Validation feature matrix shape: {X_val.shape}")


#Prepare target variable
y_val = df_feb_filtered['duration'].values

#Predict and calculate RMSE
print("Making predictions...")
y_pred_val = lr.predict(X_val)

rmse_val = mean_squared_error(y_val, y_pred_val, squared=False)
print(f"RMSE validation on data: {rmse_val:.2f}")


Februrary records after filtering: 2855951
Transforming features..
Validation feature matrix shape: (2855951, 515)
Making predictions...
RMSE validation on data: 7.81
